In [3]:
from langchain_community.document_loaders.pdf import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_openai import OpenAIEmbeddings
from langchain_community.vectorstores.chroma import Chroma

paths = [
    "documents/ApostilaLangChain.pdf",
    "documents/Explorando a API da OpenAI.pdf",
    "documents/Explorando o Universo das IAs com Hugging Face.pdf"
]

pages = []

for path in paths:
    loader = PyPDFLoader(path)
    pages.extend(loader.load())
    
splitter = RecursiveCharacterTextSplitter(
    chunk_size=500, 
    chunk_overlap=50,
    separators=["\n\n", "\n",  " ", ""]
)

documents = splitter.split_documents(pages)

embedding = OpenAIEmbeddings()

chroma_directory = "documents/chroma_vectorstore"

vector_store = Chroma.from_documents(
    documents=documents,
    embedding=embedding,
    persist_directory=chroma_directory
)

In [4]:
from langchain_openai import ChatOpenAI
from langchain.chains.retrieval_qa.base import RetrievalQA

llm = ChatOpenAI()

chat_chain = RetrievalQA.from_chain_type(
    llm=llm,
    retriever=vector_store.as_retriever(search_type="mmr"), # método MMR de retrieval
)

question = "O que é HuggingFace e como faço para acessa-lo?"

chat_chain.invoke({"query": question})

{'query': 'O que é HuggingFace e como faço para acessa-lo?',
 'result': 'O Hugging Face é uma plataforma de IA que oferece acesso a modelos de linguagem, conjuntos de dados e aplicativos para desenvolvimento de projetos de inteligência artificial. Para acessar o Hugging Face, você pode visitar o site deles em huggingface.co, onde você encontrará recursos, documentação e informações sobre como utilizar a plataforma. Além disso, você pode explorar e utilizar os recursos disponíveis por meio das bibliotecas de Python do Hugging Face.'}

### Modificando prompt da Chain

In [5]:
from langchain.prompts import PromptTemplate

new_chain_prompt = PromptTemplate.from_template("""
Utilize o contexto fornecido para responder a pergunta ao final.
Se você não sabe a resposta, apenas diga que não sabe a resposta, não tente inventar a resposta.
Utilize três frases no máximo, mantenha a resposta concisa.

Contexto: {context}

Pergunta: {question}

Resposta:
""")

chat_chain = RetrievalQA.from_chain_type(
    llm=llm,
    retriever=vector_store.as_retriever(search_type="mmr"),
    chain_type_kwargs={"prompt": new_chain_prompt},
    return_source_documents=True
)

question = "O que é HuggingFace e como faço para acessa-lo?"

response = chat_chain.invoke({"query": question})

print(response["result"])
print(response["source_documents"])

{'query': 'O que é HuggingFace e como faço para acessa-lo?',
 'result': 'O Hugging Face é uma plataforma de IA que oferece modelos, conjuntos de dados e aplicativos. Para acessá-lo, é necessário utilizar as bibliotecas de Python do Hugging Face em seus scripts. No entanto, o Hugging Face cobra apenas se você quiser hospedar um projeto privado pessoal ou da sua empresa em sua infraestrutura.',
 'source_documents': [Document(page_content='O Hugging Face cobra apenas se você quiser utilizar a infraestrutura deles para hospedar algum\nprojeto privado pessoal ou da sua empresa.\nComo usaremos o Hugging Face?\nNeste curso, vamos explorar todo o potencial que existe na plataforma do Hugging Face. Isso inclui\nentender como acessar os modelos, conjuntos de dados, e aplicativos (os chamados “Spaces”) que\nestão na plataforma. (Note que vamos usar bastante o termo “modelos” , que é um pouco mais técnico\nque IAs.)', metadata={'id': 515, 'page': 6, 'source': 'Explorando o Universo das IAs com Hug